# CardioIA — Fase 2 — Parte 2
## Classificador básico de texto: baixo risco vs. alto risco

Este notebook treina um classificador simples de Machine Learning que recebe uma frase descrevendo sintomas e prevê se a situação relatada é de **baixo risco** ou **alto risco**, simulando uma triagem clínica automatizada.

Etapas:
1. Carregar a base rotulada (`data/triagem_dataset.csv`).
2. Separar treino/teste.
3. Vetorizar as frases com **TF-IDF**.
4. Treinar um modelo de classificação (**Regressão Logística**).
5. Avaliar o desempenho (acurácia, relatório de classificação, matriz de confusão).
6. Testar o modelo com frases novas.

### Sobre a construção da base rotulada

As frases de "alto risco" em `data/triagem_dataset.csv` foram escritas usando como referência sinais de alerta reconhecidos na literatura clínica — por exemplo, dor torácica com duração maior que 20 minutos e irradiação (Manual MSD, Infarto Agudo do Miocárdio), fraqueza de um lado do corpo e dificuldade de fala (Ministério da Saúde, critério FAST para AVC) e síncope com sudorese fria (Manual MSD, Síncope). As frases de "baixo risco" foram escritas evitando deliberadamente esses sinais de alerta.

Ainda assim, isto **não é um protocolo de triagem validado**. No Brasil, a triagem de risco real em pronto-socorros e UPAs usa o **Protocolo de Manchester**, que classifica o paciente em 5 níveis de prioridade (por cor), com critérios padronizados e detalhados por queixa clínica — bem mais sofisticado do que a classificação binária (baixo/alto risco) usada aqui como exercício didático. Referência: Sociedade Brasileira de Enfermeiros de Emergência / Grupo Brasileiro de Classificação de Risco (GBCR) — ver também a descrição do protocolo em https://cemigsaude.org.br/noticias/protocolo-de-manchester-entenda-como-funciona-classificacao-de-risco-nos-prontos-socorros

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

CAMINHO_DADOS = "../data/triagem_dataset.csv"

## 1. Carregando a base rotulada

In [10]:
df = pd.read_csv(CAMINHO_DADOS)
print(f"{len(df)} frases carregadas.")
print(df["situacao"].value_counts())
df.head(10)

51 frases carregadas.
situacao
baixo risco    26
alto risco     25
Name: count, dtype: int64


,frase,situacao
0,sinto dor no peito e falta de ar,alto risco
1,estou com dor forte no peito irradiando para o...,alto risco
2,tive um desmaio subito acompanhado de suor frio,alto risco
3,sinto o coracao acelerado e falta de ar mesmo ...,alto risco
4,estou com dor no peito em aperto e nausea,alto risco
5,perdi a forca do lado esquerdo do corpo e nao ...,alto risco
6,sinto uma pressao forte no peito que nao passa...,alto risco
7,estou com falta de ar subita e os labios estao...,alto risco
8,tive palpitacoes fortes seguidas de um desmaio,alto risco
9,sinto dor no peito que piora ao respirar fundo...,alto risco


## 2. Separando treino e teste

Usamos `stratify=df["situacao"]` para manter a mesma proporção de "alto risco" e "baixo risco" nos dois conjuntos, já que a base é pequena.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    df["frase"], df["situacao"],
    test_size=0.25, random_state=42, stratify=df["situacao"]
)

print(f"Treino: {len(X_train)} frases | Teste: {len(X_test)} frases")

Treino: 38 frases | Teste: 13 frases


## 3. Vetorização TF-IDF

O `TfidfVectorizer` transforma cada frase em um vetor numérico, dando mais peso a palavras que são frequentes numa frase mas raras no restante da base (ou seja, palavras mais "informativas" para diferenciar as classes).

In [12]:
vetorizador = TfidfVectorizer()
X_train_tfidf = vetorizador.fit_transform(X_train)
X_test_tfidf = vetorizador.transform(X_test)

print(f"Vocabulário aprendido: {len(vetorizador.vocabulary_)} termos")
print(f"Formato da matriz de treino: {X_train_tfidf.shape}")

Vocabulário aprendido: 162 termos
Formato da matriz de treino: (38, 162)


## 4. Treinando o classificador

Usamos uma **Regressão Logística**, um modelo simples e um bom ponto de partida para classificação binária de texto.

In [13]:
modelo = LogisticRegression(random_state=42)
modelo.fit(X_train_tfidf, y_train)
print("Modelo treinado.")

Modelo treinado.


## 5. Avaliando o modelo

In [14]:
y_pred = modelo.predict(X_test_tfidf)

print(f"Acurácia: {accuracy_score(y_test, y_pred):.2%}\n")
print("Relatório de classificação:")
print(classification_report(y_test, y_pred))

Acurácia: 84.62%

Relatório de classificação:
              precision    recall  f1-score   support

  alto risco       0.83      0.83      0.83         6
 baixo risco       0.86      0.86      0.86         7

    accuracy                           0.85        13
   macro avg       0.85      0.85      0.85        13
weighted avg       0.85      0.85      0.85        13



In [15]:
matriz = confusion_matrix(y_test, y_pred, labels=modelo.classes_)
df_matriz = pd.DataFrame(
    matriz,
    index=[f"real: {c}" for c in modelo.classes_],
    columns=[f"previsto: {c}" for c in modelo.classes_],
)
df_matriz

,previsto: alto risco,previsto: baixo risco
real: alto risco,5,1
real: baixo risco,1,6


## 6. Testando com frases novas (fora da base de treino/teste)

In [16]:
novas_frases = [
    "sinto uma dor forte no peito e falta de ar agora",
    "tive um leve cansaco depois da caminhada",
    "meu coracao esta disparado e sinto tontura",
    "notei uma dor leve no braco depois da academia",
    "estou com o peito apertado e suando frio",
    "tive um pequeno desconforto na barriga depois do almoco",
]

novas_tfidf = vetorizador.transform(novas_frases)
previsoes = modelo.predict(novas_tfidf)
probabilidades = modelo.predict_proba(novas_tfidf)

df_novas = pd.DataFrame({
    "frase": novas_frases,
    "situacao_prevista": previsoes,
    "confianca": probabilidades.max(axis=1).round(2),
})
df_novas

,frase,situacao_prevista,confianca
0,sinto uma dor forte no peito e falta de ar agora,alto risco,0.65
1,tive um leve cansaco depois da caminhada,baixo risco,0.70
2,meu coracao esta disparado e sinto tontura,alto risco,0.59
3,notei uma dor leve no braco depois da academia,baixo risco,0.67
4,estou com o peito apertado e suando frio,alto risco,0.72
5,tive um pequeno desconforto na barriga depois ...,baixo risco,0.65


## 7. Avaliação crítica: desempenho, padrões e distorções

- **Acurácia obtida:** o modelo acerta a maior parte dos casos de teste, mesmo com uma base pequena (~50 frases) e um vetorizador simples. Isso mostra que, quando as duas classes usam vocabulário bem diferente (ex.: "leve"/"pequeno" para baixo risco vs. "forte"/"súbito"/"não passa" para alto risco), até um modelo simples de TF-IDF + Regressão Logística já separa bem os casos.
- **Padrão que o modelo aprende:** como o TF-IDF é baseado em palavras isoladas (bag-of-words), o modelo tende a aprender que certos termos ("leve", "pequeno", "forte", "súbito", "intensa") são fortes indicadores de risco — o que é útil, mas também é uma forma de **viés lexical**: uma frase de alto risco descrita sem essas palavras-gatilho ("minha visão sumiu por alguns segundos e voltei a enxergar normal") pode ser classificada erroneamente como baixo risco.
- **Tamanho da base é uma limitação real:** ~50 frases é muito pouco para um classificador de uso clínico. Com uma base tão pequena, o modelo pode estar memorizando padrões de vocabulário específicos das frases de treino, em vez de aprender uma noção mais geral de gravidade. O ideal seria validar com centenas ou milhares de frases reais (anonimizadas), revisadas por profissionais de saúde.
- **Risco de falso negativo é o mais grave neste contexto:** numa triagem clínica de verdade, classificar uma situação grave como "baixo risco" (falso negativo) é muito mais perigoso do que o contrário. Um sistema real precisaria ser ajustado (por exemplo, mudando o limiar de decisão ou usando métricas como *recall* da classe "alto risco" como prioridade) para minimizar esse tipo de erro, mesmo que isso reduza a acurácia geral.
- **Reflexão sobre IA responsável:** um classificador como este pode ajudar a *priorizar* atendimentos, mas nunca deveria *substituir* a avaliação de um profissional de saúde — ele deveria funcionar apenas como apoio, com revisão humana sempre disponível, principalmente para os casos classificados como baixo risco (que são justamente os que teriam menos atenção prioritária caso o modelo errasse).